## Every ladder game you play sheds DNA

A simulation ladder records everything. Every episode I ever played against you is a public replay: your full action stream, both observations, the resolved seed. You cannot take it back.

Many strong players publish their agents on purpose, and the ladder is better for it. Thomas Tschinkel published his [93.8 percent state router](https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router) and then his [95.5 percent replay-routing successor](https://www.kaggle.com/code/thomastschinkel/kaggriculture-95-5-win-rate-via-replay-routing), each one behaviorally identical to the submission he was fielding at that moment. yhay81's [three-day shop router](https://www.kaggle.com/code/yhay81/three-day-shop-router) seeded a whole family of derivatives. lynnsakurai's [mathematical farming approach](https://www.kaggle.com/code/lynnsakurai/farming-score-a-mathematical-approach) formalized the same trunk. Copies of these agents play on the ladder within hours of each publish, some verbatim, some repackaged. And the arms race around copying is itself public: boatlee's [V14 release](https://www.kaggle.com/code/boatlee/84-84-base-public-holdout-v14-clone-preemption) ships replay clustering, action-by-action route recovery, and a near-clone preemption controller. The difference in this page is the target and the unit of capture: V14 reconstructs a strong SCHEDULE from clustered replays and defends against near-clones of itself; here the captured object is a specific adaptive agent's entire repertoire and branching behavior, rebuilt under a dispatch of my own and validated as a fielded copy.

This notebook asks the question that follows for everyone else: given only your public REPLAYS, never your code, **can I clone you anyway?**

The DNA framing is not new to this series: [A DNA Test for Agents](https://www.kaggle.com/code/destbreso/a-dna-test-for-agents) showed that an agent's recorded actions identify it the way a genome does, well enough to tell relatives from strangers across the whole ladder. This notebook asks the question that test makes unavoidable: if the replays are the DNA, can I grow the organism back? We all watched Jurassic Park, so let us call the procedure what it is: cloning. And since I intend to run with the bit, every lab that hatches its first specimen owes it a species name; mine is walking the ladder right now, and in my notes it goes by Clonosaurus branchii. Your replays are the mosquito in the amber, and the question is only whether they hold enough DNA.

The answer I found is: it depends on **a single measurable property** of your agent, and if you have it, I can rebuild a working copy of you in an afternoon without ever learning how you think. I did it to a top-10 agent this week. The copy passed every validation gate I own, including a 96 percent win rate across the full grid of worlds and opponent strengths.

The cloned agent was never published by its team. For exactly that reason I will not name it, show its code, or cite its rating history. The method is the point, and the last section is for defenders: the property that makes you clonable is cheap to remove.


## How I got here, in published steps

This is not a trick I found in one sitting. It is the latest step of a line of work, and each step is public.

First came the observation that a recorded episode is a complete, replayable object: I published the [top-10 replay streams](https://www.kaggle.com/datasets/destbreso/kaggriculture-top10-replay-streams) as a dataset, and the companion study [Mutants at the Top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) measured which fraction of a top agent's stream you are even allowed to change: the genetic potential of a replay, which is where the DNA framing in this notebook was born.

Second came the measurement discipline, because my first instruments lied to me and the retractions taught more than the results: [Measure Twice, Submit Once](https://www.kaggle.com/code/destbreso/measure-twice-submit-once-know-your-error) documents the evaluation protocol every number in this notebook passed through, including the band-by-world matrix used as the final exam below. The [donor agents snapshot](https://www.kaggle.com/datasets/destbreso/kaggriculture-donor-agents-20260902) holds the credited, license-checked community agents that protocol evaluates against, with behaviour families measured by game rather than assumed by name.

Underneath all of it sits the piece of infrastructure that made the deeper experiments affordable at all: a C++ port of the game engine, published in [kagsim, the engine at 2000 episodes per second](https://www.kaggle.com/code/destbreso/kagsim-the-engine-at-2000-episodes-per-second) and pushed further in [From 1 to 24k episodes a second](https://www.kaggle.com/code/destbreso/from-1-to-24k-episodes-a-second). Before it, a single evaluation was an evening; after it, massive cross-panels of thousands of games, exhaustive sweeps over all 64 shop worlds, and live-observation pairing, where a candidate plays turn by turn against a recorded rival with full observations instead of a canned tape, became routine. Every table in this notebook exists because those three capabilities got cheap.

In between sit the two diagnostic pieces this notebook leans on directly. [A DNA Test for Agents](https://www.kaggle.com/code/destbreso/a-dna-test-for-agents) established identity-by-stream: the hashing that this notebook's capturability test is built from. And [X-ray your agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent) turned replays into a diagnostic panel that classifies how an agent plays, which is where the class this notebook hunts first showed up as a distinct silhouette: the BRANCHER, one fixed opening committing to a continuation at a decision point.

Last is this notebook: from replaying one fixed route, to routing a family of recorded streams, to cloning an unpublished brancher through its outputs. Each step reuses the previous one's machinery, and none of it needed anything beyond what the ladder makes public.


## Step 1: the capturability test

The top of a competitive ladder cannot usually be copied as a fixed route. I measured that early: two episodes that differ prove an agent adapts, and almost every strong team's episodes differ.

But there are kinds of different. The test that matters is the size of the repertoire: pull a handful of an agent's episodes, hash the full action stream of each, and count distinct streams.


In [ ]:
import hashlib, json

def stream_hash(actions):
    """One line of DNA per episode: the hash of everything the agent did."""
    return hashlib.md5(json.dumps(actions, sort_keys=True).encode()).hexdigest()[:8]

# What the test returns for three real agents (8 episodes each, distinct hashes):
#   agent A: 8 distinct of 8   -> fully reactive or a huge repertoire; not capturable this way
#   agent B: 8 distinct of 8   -> same
#   agent C: 4 distinct of 8   -> a FINITE REPERTOIRE; this is the target
print("distinct streams over 8 episodes: 8 = safe, small = clonable")

In [ ]:
# The target's full public history, crawled and hashed: 238 episodes.
repertoire = {
    "stream_01": 70, "stream_02": 36, "stream_03": 36, "stream_04": 24,
    "stream_05": 16, "stream_06": 15, "stream_07": 9,  "stream_08": 6,
    # ... 11 more streams with 1-5 episodes each
}
total, distinct = 238, 19
top6 = sum(sorted(repertoire.values(), reverse=True)[:6])
print(f"{total} episodes -> {distinct} distinct streams")
print(f"top 6 streams cover {top6}/{total} = {top6/total:.0%} of everything it ever played")

Most of the top ten returned 8 of 8: eight episodes, eight different games, nothing to hold on to. One returned four.

Deepened to its full public history of 238 episodes, that agent produced only 19 distinct streams, and the top six covered 83 percent of all its games. One fixed opening, then a branch, then a finite menu of continuations.

That distribution is the whole diagnosis, and it lands the agent in a class my [X-ray notebook](https://www.kaggle.com/code/destbreso/x-ray-your-agent) already knew how to name: a BRANCHER. One fixed opening, a decision point, a finite menu. An agent with a finite repertoire is not adaptive in the way that protects you; it is **a lookup table wearing an adaptive coat**, and a lookup table can be stolen through its outputs. The X-ray classifies; this notebook is what the classification is FOR.

Two traps in this step, both of which I paid for before writing this. 

The diagnostic itself has published precedent: Jeff Marc E. Cadet ran this exact count on a top-5 team and reported the negative verdict, [19 distinct builds across 19 replays, not a tape](https://www.kaggle.com/code/jeffmarcecadet/reverse-engineering-rank-5-thunder-thunder), commands included. What this notebook adds is the other branch: what to do when the count comes back small, which is the harvest, the rebuild, and the validation of a fielded copy.
**Two episodes prove variation but never fixity**, so classify on many or not at all. And the distance between one agent and a known family is a per-episode random variable, not a constant: I watched one independent router read 234 modified turns in one episode and 590 in another against the same reference, and a single-episode cutoff misfiled it.


## Step 2: find the branch point

If the repertoire is finite, the next question is where the branching happens, because that names the information the agent conditions on.

Take two episodes with different streams and find the first turn where the actions diverge. For my target the answer was surprising: not at any of the scheduled world reveals, but a handful of turns after the second one, at the exact moment the market state first reflects what the OPPONENT did with its opening.

The agent was not branching on the world. It was branching on **the rival's footprint in the shared market**: an opponent-class router. I confirmed this by training a one-split rule per world on the state at the branch turn and validating on held-out episodes: the world alone predicts the branch 67 percent of the time, and the residual splits on features that encode the rival's opening.


## Step 3: the harvest, and why I never learned its rule

67 percent held-out accuracy is not a replica. Here is the insight that makes the method work anyway: **I do not need the rule.**

The rule chooses among continuations. **The continuations themselves are data**, sitting in the replays. So the harvest is:

1. Crawl the agent's full public history: action stream, world sequence, resolved seed, per episode.
2. Deduplicate by stream hash. My 238 crawled episodes collapsed to 19 distinct tapes: the genome was almost all redundant copies, which is also why the raw corpus was 12 times heavier than the information it contained.
3. Rebuild under MY OWN dispatch: their tapes, keyed by the world sequence with longest-prefix matching, driven by the routing chassis I already trust. Their opening stays attached to their continuations, because a continuation spliced onto a foreign opening inherits a farm it did not build.

The clone that results does not think like the original. It routes like me while playing like them, and the validation decides whether that hybrid is any good.

One engineering aside that mattered: before deduplication the raw corpus loaded at 878 MB of process memory per instance, and an evaluation arena spinning several instances took the whole machine down twice. After deduplication the same behavior loads at 80 MB. If you harvest, dedupe first; **the genome is smaller than the organism**.

The branch rule itself I also tried to learn, as a control: a per-world lookup on the public state at the branch turn predicts the chosen continuation at 67 percent on held-out episodes, and the residual error concentrates on features that encode the opponent's opening. Good enough to confirm WHAT the agent conditions on, nowhere near good enough to replicate it, and the harvest made it unnecessary.


## Step 4: the final exam

My standing gate for any candidate is a matrix: win rate against recorded rivals stratified by strength band, crossed with every world the game can deal. Flat rows mean robustness; weak cells name exactly where the agent will bleed if fielded.

The clone's matrix, 570 games over 64 worlds and 7 strength bands, up to the strongest recorded opposition I hold:


### How the exam is built, because the instrument is half the result

The opponents are not synthetic and not chosen for strength. Every rival in the table is a RECORDED ladder stream: the full action sequence a real submission played in a real episode, replayed exactly. I learned to insist on this the expensive way: an earlier version of my evaluation used a panel of the five strongest public agents, and its ranking of candidates INVERTED against the ladder's own verdict, because **the ladder does not pay for beating the best five, it pays for beating whoever it deals you**. [Measure Twice, Submit Once](https://www.kaggle.com/code/destbreso/measure-twice-submit-once-know-your-error) documents that inversion and the protocol that replaced it.

The bands stratify those recorded rivals by the rating their team held when the stream was captured, from ordinary submissions near 900 up to streams recorded from the current top 15. A rating is a timestamp, not a property, so the pool is rebuilt from fresh episodes every day or two; the field's composition has a measured half-life of about two days.

The worlds are the 64 ordered draws of the first two shops. That axis is not arbitrary: the second shop is revealed by the time capital is committed, the routing decisions of most of the field branch on exactly that prefix, and 64 columns is the finest world partition the reveal calendar pays for. The same reference covers why two shops and not one or three.

The pairing rule: each cell replays the rival's own recorded episode, at its own resolved seed and world, with the candidate substituted into one seat and the rival's stream driving the other, both seats where the pool allows. The engine is my C++ engine-exact simulator, the same one from the lineage section: it reproduces recorded final banks to the dollar, which is what makes a recorded stream a valid baseline in the first place, and every game in the table runs in live-pairing mode, the candidate receiving full observations turn by turn rather than replaying a canned script of its own. At its throughput the whole 570-game exam, plus the 3,328-game donor cross behind it, fits in a coffee break, which is the practical difference between a validation you run once and a gate you run on every candidate.

One honest limitation carries over from that lineage: a recording cannot react. Against fixed and repairing rivals the interaction bias of replay-based scoring measured under one percent of a typical margin, but against a genuinely adaptive rival the replay plays a different game than the live agent would, so the table's strongest cells are read as ordering evidence, and the live ladder below is the only judge that needs no caveat.


The full table, 64 worlds by 7 opponent-strength bands, wins/games per cell. Rows are the first two shops the world reveals; the strongest band is recorded play from the current top 15.

```text
world                 ~900    ~1500    ~1650    ~2130    ~2250    ~2550    top15     TOTAL
PIZZA_S|BAKERY         2/2      2/2      2/4        ·        ·      2/2    12/12  20/22 91%
SMOOTHI|YARN_ST          ·        ·      6/6      2/2        ·      2/2    10/10  20/20 100%
BRUNCH_|PET_CAF          ·      1/2      6/6      2/2      2/2        ·      6/6  17/18 94%
PET_CAF|PET_CAF          ·        ·      4/6      2/2      2/2      2/2      4/4  14/16 88%
PET_CAF|YARN_ST          ·        ·      2/2        ·      2/2      6/6      5/6  15/16 94%
BRUNCH_|BAKERY           ·      2/2        ·      2/2        ·      4/4      8/8  16/16 100%
PET_CAF|FARMERS        2/2      2/2        ·      2/2        ·      6/6      4/4  16/16 100%
YARN_ST|YARN_ST          ·        ·      2/4      2/2        ·      2/2      8/8  14/16 88%
ICE_CRE|ICE_CRE        4/4        ·      4/6        ·        ·        ·      4/4  12/14 86%
ICE_CRE|YARN_ST        2/2      2/2      6/6      2/2      2/2        ·        ·  14/14 100%
ICE_CRE|BRUNCH_          ·      2/2        ·      2/2        ·        ·     6/10  10/14 71%
SMOOTHI|PET_CAF          ·      2/2      4/4      2/2      2/2      2/2        ·  12/12 100%
PET_CAF|BAKERY           ·        ·      2/2      2/2      2/2      2/2      4/4  12/12 100%
YARN_ST|SMOOTHI          ·        ·      4/4      2/2      2/2        ·      4/4  12/12 100%
BAKERY|PET_CAF          ·        ·      2/2      2/2        ·      2/2      6/6  12/12 100%
BAKERY|BAKERY           ·        ·      2/2      2/2        ·      4/4      4/4  12/12 100%
FARMERS|BAKERY           ·        ·      4/4      2/2      2/2      2/4        ·  10/12 83%
PIZZA_S|PIZZA_S          ·        ·      2/2      2/2      2/2      4/4      2/2  12/12 100%
ICE_CRE|BAKERY         4/4        ·        ·        ·        ·      4/4      2/2  10/10 100%
BAKERY|FARMERS        2/2      2/2      2/2        ·        ·      4/4        ·  10/10 100%
FARMERS|PET_CAF          ·        ·      2/2        ·        ·      4/4      4/4  10/10 100%
YARN_ST|PET_CAF        4/4        ·      2/2      2/2      2/2        ·        ·  10/10 100%
BRUNCH_|PIZZA_S        2/2        ·      2/2      2/2        ·      4/4        ·  10/10 100%
YARN_ST|ICE_CRE          ·      2/2        ·        ·        ·      4/4      4/4  10/10 100%
YARN_ST|PIZZA_S        2/2        ·        ·      2/2        ·      4/4      2/2  10/10 100%
PET_CAF|SMOOTHI          ·        ·      6/6        ·      2/2        ·        ·    8/8 100%
PIZZA_S|PET_CAF          ·        ·        ·      2/2        ·      4/4      2/2    8/8 100%
ICE_CRE|PET_CAF        2/2      2/2        ·        ·        ·      2/2      2/2    8/8 100%
FARMERS|FARMERS          ·      2/2        ·      2/2        ·      4/4        ·    8/8 100%
PIZZA_S|FARMERS          ·      2/2      2/2        ·        ·      2/2      2/2    8/8 100%
YARN_ST|BRUNCH_          ·        ·      2/2      2/2        ·        ·      4/4    8/8 100%
FARMERS|PIZZA_S        6/6        ·        ·        ·        ·        ·      2/2    8/8 100%
FARMERS|BRUNCH_        4/4        ·      2/2        ·        ·        ·      2/2    8/8 100%
BAKERY|YARN_ST        2/2        ·      2/2        ·        ·      2/2      2/2    8/8 100%
SMOOTHI|BAKERY           ·        ·      2/2        ·        ·      2/2      4/4    8/8 100%
BRUNCH_|YARN_ST          ·        ·      4/4      2/2        ·      2/2        ·    8/8 100%
SMOOTHI|SMOOTHI          ·        ·      2/2        ·      2/2      2/2      2/2    8/8 100%
BAKERY|SMOOTHI          ·        ·      2/2        ·        ·      4/4      2/2    8/8 100%
PIZZA_S|BRUNCH_          ·        ·      2/2        ·        ·        ·      6/6    8/8 100%
PET_CAF|ICE_CRE          ·        ·      2/2        ·        ·      2/2      4/4    8/8 100%
YARN_ST|FARMERS        2/2        ·      2/2        ·      2/2      2/2        ·    8/8 100%
FARMERS|SMOOTHI          ·        ·      2/2      2/2        ·        ·      4/4    8/8 100%
PIZZA_S|SMOOTHI          ·      2/2      2/2        ·        ·        ·      4/4    8/8 100%
BRUNCH_|ICE_CRE        2/2        ·        ·      2/2        ·        ·      2/2    6/6 100%
BAKERY|PIZZA_S          ·        ·        ·        ·        ·      2/2      4/4    6/6 100%
PET_CAF|PIZZA_S        2/2        ·      2/2        ·        ·        ·      2/2    6/6 100%
YARN_ST|BAKERY           ·      4/4      2/2        ·        ·        ·        ·    6/6 100%
BRUNCH_|BRUNCH_          ·        ·        ·      4/4        ·      2/2        ·    6/6 100%
ICE_CRE|SMOOTHI        2/2      4/4        ·        ·        ·        ·        ·    6/6 100%
BRUNCH_|FARMERS          ·        ·      2/4        ·      0/2        ·        ·    2/6 33%
BAKERY|ICE_CRE        2/2      2/2        ·      2/2        ·        ·        ·    6/6 100%
FARMERS|YARN_ST          ·      2/2        ·      2/2        ·      2/2        ·    6/6 100%
PIZZA_S|YARN_ST          ·      2/2      2/2        ·        ·        ·        ·    4/4 100%
PIZZA_S|ICE_CRE          ·        ·      4/4        ·        ·        ·        ·    4/4 100%
SMOOTHI|BRUNCH_          ·        ·        ·        ·        ·      2/2      2/2    4/4 100%
PET_CAF|BRUNCH_          ·      0/2        ·        ·        ·        ·      2/2    2/4 50%
ICE_CRE|FARMERS          ·        ·        ·      2/2        ·      2/2        ·    4/4 100%
ICE_CRE|PIZZA_S          ·        ·        ·      2/2        ·      2/2        ·    4/4 100%
SMOOTHI|ICE_CRE          ·      2/2        ·        ·        ·      2/2        ·    4/4 100%
SMOOTHI|PIZZA_S          ·        ·      2/2        ·        ·        ·        ·    2/2 100%
SMOOTHI|FARMERS          ·        ·      2/2        ·        ·        ·        ·    2/2 100%
FARMERS|ICE_CRE          ·        ·        ·        ·        ·      2/2        ·    2/2 100%
BRUNCH_|SMOOTHI          ·        ·        ·        ·        ·        ·      0/2    0/2  0%
------------------------------------------------------------------------------------------
BAND:                 100%      93%      92%     100%      93%      98%      96%   GLOBAL 546/570 = 96%```

How to read it: a dot is an empty pairing (no recorded rival of that strength in that world), the TOTAL column is the row's aggregate, and the BAND row at the bottom is each band's aggregate.


**96 percent global (546 of 570), no band under 92**, and the highest score this gate has ever produced across every agent I have built or borrowed.

The rest of the battery, because one instrument is never enough and my own retractions taught me to demand order agreement between populations:

Against the pool of rivals distilled from the specific submissions that beat my live agents this same week, the clone wins 68 percent with a median margin of +2,099, where my previous best line wins 11 percent with a median of -1,067 on the identical pool.

Against a panel of 25 credited community donors over all 64 worlds and both seats, 3,328 games per challenger, the clone reads 97.9 percent against my incumbent's 97.0: same ordering, second population, agreement.

Head to head, the clone beats both of my own fielded agents 38 wins to 10 with a median margin of +3,001. Whatever it meets on the ladder, it will not be farmed by its own siblings.

Macro shape, read directly from live simulation: land expansions on days 6 and 11 where my incumbent buys on 7 and 13, stranded end-of-game stock of exactly zero dollars at median, and a bank median above my strongest reference column.

The weak cells cluster on worlds containing one particular shop, which is not a flaw of the method: it is the original's own blind spot, faithfully cloned along with everything else. **DNA copies the disease too.**


## The clone is live

This is not a paper exercise. Clonosaurus branchii is my current submission 56101944, fielded the same day the harvest finished, credited in its submission message to the team whose play it carries.

That makes this notebook **a pre-registered live experiment**: the offline battery above predicts a specific class of performance, the ladder will spend the next days agreeing or disagreeing, and both outcomes are informative. If the copy converges near its source's band, cloning-from-replays transfers. If it decays fast while the unnamed original holds, the difference is the branch rule I chose not to learn: the one piece of the organism the DNA did not carry.

Either way the measurement is public, which is the standard this whole series holds itself to.


## The defender's section, which is the real point

If you run an agent on a public-replay ladder, the capturability test works on you tonight. Whether it succeeds is your choice:

An agent whose whole game is a deterministic function from public state to a finite set of continuations is **clonable through its outputs**, full stop. No obfuscation of the code changes this, because the code is not what gets stolen.

The cheap defense is **entropy**: a per-episode source of variation, even a tiny one, drawn from something the harvester cannot condition on. A salt over equivalent continuations breaks stream-hash clustering, inflates the apparent repertoire past the crawl budget, and turns the harvest into noise. It costs a few points against nobody and denies the copy entirely.

The expensive defense is genuine adaptivity: condition your play on the opponent finely enough that your repertoire is effectively unbounded. The one agent at the very top of my ladder that varies its play before the first reveal is, by this method, untouchable.

I built the clone, validated it, and fielded it. What happens to it on the live ladder is the one measurement this notebook cannot contain yet, because ratings take days and fields drift in two. But the method is now a procedure with a checklist, and the next finite repertoire I meet is an afternoon of work away from being mine.


## Reproduce it yourself

Everything below runs against endpoints the platform serves publicly, needs no authentication, and is the complete toolchain for steps 1 through 3. The trigger cell at the bottom is aimed, on purpose, at the submission you should test FIRST: your own. Put your submission id in, and thirty seconds later you know whether the mosquito in the amber is you. This page deliberately runs it on nobody, because the results worth publishing are the macro tables above, not any team's identity.

Validation is the part you must bring yourself: an evaluation stack you trust, built the way [Measure Twice, Submit Once](https://www.kaggle.com/code/destbreso/measure-twice-submit-once-know-your-error) prescribes, because a harvest without a gate is just a pile of tapes.


In [ ]:
import collections, hashlib, json, time, urllib.request

LIST_URL = "https://www.kaggle.com/api/i/competitions.EpisodeService/ListEpisodes"
REPLAY_URL = "https://www.kaggleusercontent.com/episodes/{id}.json"


def _post(url, payload):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=60).read())


def episodes_of(submission_id):
    """Every completed episode the ladder holds for one submission."""
    eps = _post(LIST_URL, {"submissionId": int(submission_id)}).get("episodes", [])
    return sorted((e for e in eps if e.get("state") == "COMPLETED"),
                  key=lambda e: e.get("endTime") or "")


def stream_of(episode, submission_id):
    """The agent's full action stream from one public replay, plus the world."""
    rep = json.loads(urllib.request.urlopen(
        REPLAY_URL.format(id=episode["id"]), timeout=90).read())
    steps = rep["steps"]
    seat = next(i for i, a in enumerate(episode["agents"])
                if a.get("submissionId") == int(submission_id))
    acts = [steps[t][seat].get("action") or {} for t in range(1, len(steps))]
    town = (steps[-1][0].get("observation", {}) or {}).get("town") or {}
    return acts, town.get("unlocked_shops") or [], (rep.get("info") or {}).get("seed")


def repertoire(submission_id, sample=16, pause=0.3):
    """Step 1, the capturability test: distinct full streams over a sample.
    8 of 8 distinct means walk away; a concentrated handful means a target."""
    counts = collections.Counter()
    for e in episodes_of(submission_id)[-sample:]:
        acts, _, _ = stream_of(e, submission_id)
        counts[hashlib.md5(json.dumps(acts, sort_keys=True).encode()).hexdigest()[:8]] += 1
        time.sleep(pause)
    return counts


def branch_point(acts_a, acts_b):
    """Step 2: the first turn two streams disagree names what the rule reads."""
    return next((t for t in range(min(len(acts_a), len(acts_b)))
                 if acts_a[t] != acts_b[t]), None)


def harvest(submission_id, pause=0.3):
    """Step 3: the deduplicated genome, keyed for a world-prefix dispatch.
    Feeding it to a router and validating the result is your half of the work."""
    tapes, keys = {}, collections.defaultdict(list)
    for e in episodes_of(submission_id):
        acts, shops, seed = stream_of(e, submission_id)
        h = hashlib.md5(json.dumps(acts, sort_keys=True).encode()).hexdigest()[:10]
        tapes.setdefault(h, acts)
        keys[tuple(shops[:2])].append(h)
        time.sleep(pause)
    return {"tapes": tapes, "world_index": dict(keys)}


def am_i_clonable(submission_id, sample=16):
    """The self-test: run the capturability diagnosis on YOUR OWN submission
    and get the verdict the harvester would reach about you."""
    dist = repertoire(submission_id, sample=sample)
    n, k = sum(dist.values()), len(dist)
    top = max(dist.values()) if dist else 0
    print(f"{n} episodes sampled, {k} distinct streams, largest cluster {top}")
    if k == n:
        print("VERDICT: not capturable this way. Every episode is a different game;")
        print("there is no finite repertoire to steal. (A larger sample can still revise this.)")
    elif k <= max(2, n // 3):
        print("VERDICT: CLONABLE. A concentrated finite repertoire: a harvester gets")
        print("a working copy of you in an afternoon. The defense section above is for you.")
    else:
        print("VERDICT: borderline. Partial clustering; deepen the sample before relaxing,")
        print("because two episodes prove variation and never fixity.")
    return dist

# The trigger. Test yourself first; whoever you point it at, the verdict is real:
MY_SUBMISSION_ID = None
if MY_SUBMISSION_ID:
    am_i_clonable(MY_SUBMISSION_ID)
else:
    print("set MY_SUBMISSION_ID to your own submission and run the self-test")

## Where I stand on this, plainly

Is this an honest way to play? **Yes.** Everything here reads only what the platform makes public by design, the fielded copy credits its source in its own submission message, and this notebook documents the method in the open, defenses included.

Does it violate the competition's rules? **No.** Public replays are public, and building on the field's published and recorded work is how this ladder has always moved; half the leaderboard descends from notebooks their authors wanted used. My own standing rule, kept in every agent I have ever fielded: the original authors are always credited, whether I build on their published work or on their recorded replays, in the module itself and in the submission message the ladder displays.

Would I want to WIN this way? **Definitely not, and the sharper point is that I do not believe you can.** Copies depreciate: I have watched byte-identical agents shed hundreds of rating points in days as the field learns them, my own fielded copies included, and [Mutants at the Top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay) already showed how little of a strong replay is even yours to change. **A clone buys a position and a measurement window; it does not buy the next move.** The team that owns the generator makes the next move for free.

Which is why **the largest share of merit here belongs to the people building the original generative material**: the agents worth cloning, and the published ones seeding the whole ecosystem. They are doing the hard part.

And still: this result is original, the method is real, the validation held, and chasing an unpublished top-10 agent through its fossil record until a working copy of it walked was, frankly, fun. That is my whole defense, and I think it is enough.

And the part I am most curious about: if you replicate this on your end, against your own choice of target, tell me what you found in the comments. Comparing harvests would be the best possible sequel to this page. And if the self-test above served you the other way, as the tool that told you to go salt your agent before someone like me came around, an upvote helps the next defender find it too.
